# CloudTCP: evaluate a saved baseline

This notebook calculates scores from the prediction TIFFs already saved by a training run. It does not retrain the model or need a GPU.

Provide the original `dataset_split.zip` and the run's results ZIP. By default, the notebook looks in `MyDrive/CloudTCP/runs/`. You can also upload both ZIPs directly.

The output includes per-class scores, class rankings, validation-file references, and the original logs. It also checks that comparing the baseline with itself gives zero change.


## 1. Install rasterio


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio==1.4.4"])


## 2. Load the evaluation code

The helper code is included below.


In [ ]:
from pathlib import Path
import json
TOOL_DIR = Path("/content/cloudtcp_w4_baseline_tools")
TOOL_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_FILES = json.loads('{"baseline_dataset_core.py": "\\"\\"\\"Dataset reader and metrics copied from the existing CloudTCP baseline.\\nSource: CloudTCP_SegFormer.py, SHA-256\\nb983ec207318713f0d06e41dc3722cc0e149a04611801f35c3340bdf891b5fe8.\\nOnly imports were reduced; the selected function bodies are unchanged.\\nNo model or training is performed by this module.\\n\\"\\"\\"\\nfrom pathlib import Path\\nfrom collections import Counter\\nimport csv\\nimport hashlib\\nimport json\\nimport zipfile\\nimport numpy as np\\nimport rasterio\\nIGNORE = 255\\n\\ndef sha256(path):\\n    digest = hashlib.sha256()\\n    with Path(path).open(\\"rb\\") as f:\\n        for block in iter(lambda: f.read(1024 * 1024), b\\"\\"):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\ndef extract_dataset(zip_path, workspace):\\n    \\"\\"\\"Unpack into a content-addressed local folder; keep Drive I/O out of training.\\"\\"\\"\\n    dest = Path(workspace) / (\\"dataset_\\" + sha256(zip_path)[:12])\\n    dest.mkdir(parents=True, exist_ok=True)\\n    with zipfile.ZipFile(zip_path) as archive:\\n        for item in archive.infolist():\\n            if \\"__MACOSX\\" in Path(item.filename).parts:\\n                continue\\n            target = (dest / item.filename).resolve()\\n            if not target.is_relative_to(dest.resolve()):\\n                raise ValueError(\\"ZIP contains an unsafe path.\\")\\n            archive.extract(item, dest)\\n    roots = [p.parent for p in dest.rglob(\\"pipeline_settings.json\\")\\n             if (p.parent / \\"manifest.csv\\").exists()]\\n    if len(roots) != 1:\\n        raise ValueError(\\"Expected one dataset folder containing manifest.csv and pipeline_settings.json.\\")\\n    return roots[0]\\n\\ndef _inside(root, name):\\n    path = (Path(root) / name).resolve()\\n    if not path.is_relative_to(Path(root).resolve()) or not path.is_file():\\n        raise ValueError(f\\"Missing/invalid dataset path: {name}\\")\\n    return path\\n\\ndef read_pair(row, bands):\\n    with rasterio.open(row[\\"image_path\\"]) as src:\\n        if len(bands) != 3 or min(bands) < 1 or max(bands) > src.count:\\n            raise ValueError(\\"This RGB baseline needs three valid, 1-based image band indices.\\")\\n        if any(src.dtypes[b - 1] != \\"uint8\\" for b in bands):\\n            raise ValueError(\\"Expected uint8 imagery; specify scaling before using another dtype.\\")\\n        image = src.read(bands).transpose(1, 2, 0)\\n        valid = np.all(src.read_masks(bands) > 0, axis=0)\\n        profile = src.profile.copy()\\n        bounds = tuple(src.bounds)\\n        source_bands = src.count\\n    with rasterio.open(row[\\"mask_path\\"]) as src:\\n        if src.count != 1 or not np.issubdtype(np.dtype(src.dtypes[0]), np.integer):\\n            raise ValueError(f\\"Expected a single-band integer mask: {row[\'chip\']}\\")\\n        if (src.height, src.width) != image.shape[:2] or src.crs != profile[\\"crs\\"]:\\n            raise ValueError(f\\"Image/mask size or CRS mismatch: {row[\'chip\']}\\")\\n        if not src.transform.almost_equals(profile[\\"transform\\"]):\\n            raise ValueError(f\\"Image/mask transform mismatch: {row[\'chip\']}\\")\\n        if src.nodata is not None and src.nodata != IGNORE:\\n            raise ValueError(\\"Mask nodata must be 255 or unset. Class 0 is annotated background.\\")\\n        labels = src.read(1).astype(np.int64)\\n    invalid_ids = set(np.unique(labels).tolist()) - {0, 1, 2, 3, 4, IGNORE}\\n    if invalid_ids:\\n        raise ValueError(f\\"Unexpected mask IDs {invalid_ids}: {row[\'chip\']}\\")\\n    labels[~valid] = IGNORE\\n    if not np.any(labels != IGNORE):\\n        raise ValueError(f\\"No supervised pixels in {row[\'chip\']}\\")\\n    return dict(image=image, mask=labels, valid=valid, profile=profile,\\n                bounds=bounds, source_bands=source_bands, row=row)\\n\\ndef inspect_dataset(root, input_bands=(1, 2, 3)):\\n    root = Path(root)\\n    settings = json.loads((root / \\"pipeline_settings.json\\").read_text())\\n    classes = {int(k): v for k, v in settings[\\"classes\\"].items()}\\n    if set(classes) != set(range(5)) or settings.get(\\"ignore_label\\") != IGNORE:\\n        raise ValueError(\\"Expected background=0, foreground classes=1..4, ignore=255.\\")\\n    names = [classes[i] for i in range(5)]\\n    with (root / \\"manifest.csv\\").open(newline=\\"\\") as f:\\n        rows = list(csv.DictReader(f))\\n    if not rows or not {\\"chip\\", \\"split\\", \\"image\\", \\"mask\\"}.issubset(rows[0]):\\n        raise ValueError(\\"Manifest must include chip, split, image and mask columns.\\")\\n    samples, seen, digest = [], set(), hashlib.sha256()\\n    digest.update((root / \\"manifest.csv\\").read_bytes())\\n    digest.update((root / \\"pipeline_settings.json\\").read_bytes())\\n    for row in rows:\\n        if row[\\"chip\\"] in seen or row[\\"split\\"] not in {\\"train\\", \\"val\\"}:\\n            raise ValueError(\\"Duplicate chip name or invalid split.\\")\\n        seen.add(row[\\"chip\\"])\\n        row[\\"image_path\\"] = _inside(root, row[\\"image\\"])\\n        row[\\"mask_path\\"] = _inside(root, row[\\"mask\\"])\\n        for key in (\\"image_path\\", \\"mask_path\\"):\\n            digest.update(sha256(row[key]).encode())\\n        samples.append(read_pair(row, list(input_bands)))\\n    splits = {split: [s for s in samples if s[\\"row\\"][\\"split\\"] == split]\\n              for split in (\\"train\\", \\"val\\")}\\n    if not all(splits.values()):\\n        raise ValueError(\\"Both train and validation need image/mask pairs.\\")\\n    if len({s[\\"image\\"].shape for s in samples}) != 1:\\n        raise ValueError(\\"This baseline expects equal chip dimensions; it never silently resizes labels.\\")\\n    group_overlap = sorted({s[\\"row\\"].get(\\"group\\") for s in splits[\\"train\\"]}\\n                           & {s[\\"row\\"].get(\\"group\\") for s in splits[\\"val\\"]} - {None, \\"\\"})\\n    overlap = []\\n    for a in splits[\\"train\\"]:\\n        for b in splits[\\"val\\"]:\\n            if a[\\"profile\\"][\\"crs\\"] != b[\\"profile\\"][\\"crs\\"]:\\n                raise ValueError(\\"Mixed CRSs require a common-CRS leakage check first.\\")\\n            aa, bb = a[\\"bounds\\"], b[\\"bounds\\"]\\n            dx = min(aa[2], bb[2]) - max(aa[0], bb[0])\\n            dy = min(aa[3], bb[3]) - max(aa[1], bb[1])\\n            if dx > 1e-4 and dy > 1e-4:\\n                overlap.append([a[\\"row\\"][\\"chip\\"], b[\\"row\\"][\\"chip\\"]])\\n    if group_overlap or overlap:\\n        raise ValueError(f\\"Split leakage: shared groups={group_overlap}, overlapping chips={overlap}\\")\\n    counts = {split: np.zeros(5, dtype=np.int64) for split in splits}\\n    table = []\\n    for split, ss in splits.items():\\n        for sample in ss:\\n            y = sample[\\"mask\\"]\\n            counts[split] += np.bincount(y[y != IGNORE], minlength=5)\\n    if np.any(counts[\\"train\\"][1:] == 0):\\n        raise ValueError(\\"At least one foreground class has no training labels.\\")\\n    for cls, name in enumerate(names):\\n        row = {\\"id\\": cls, \\"class\\": name}\\n        for split, ss in splits.items():\\n            row[f\\"{split}_pixels\\"] = int(counts[split][cls])\\n            row[f\\"{split}_percent\\"] = float(100 * counts[split][cls] / counts[split].sum())\\n            row[f\\"{split}_chips\\"] = sum(bool(np.any(s[\\"mask\\"] == cls)) for s in ss)\\n        table.append(row)\\n    audit = dict(dataset_fingerprint=digest.hexdigest(),\\n                 n_chips={s: len(v) for s, v in splits.items()},\\n                 class_names=names, input_bands=list(input_bands),\\n                 shape=list(samples[0][\\"image\\"].shape), source_bands=samples[0][\\"source_bands\\"],\\n                 label_status=dict(Counter(r.get(\\"label_status\\", \\"unknown\\") for r in rows)),\\n                 crs=str(samples[0][\\"profile\\"][\\"crs\\"]),\\n                 exported_pixel_spacing=list(samples[0][\\"profile\\"][\\"transform\\"])[0:6:4],\\n                 class_counts={k: v.tolist() for k, v in counts.items()},\\n                 ignored_pixels=sum(int((s[\\"mask\\"] == IGNORE).sum()) for s in samples),\\n                 group_overlap=group_overlap, overlapping_train_val_pairs=overlap,\\n                 class_report=table,\\n                 caveat=\\"Label agreement with provided annotations; auto_draft is not verified ground truth.\\")\\n    # The supplied UTM raster uses metres. Pixel sampling is not verified native sensor GSD.\\n    audit[\\"exported_pixel_spacing\\"] = [abs(samples[0][\\"profile\\"][\\"transform\\"].a),\\n                                       abs(samples[0][\\"profile\\"][\\"transform\\"].e)]\\n    return dict(root=root, settings=settings, samples=samples, splits=splits,\\n                counts=counts, names=names, audit=audit)\\n\\ndef metrics_from_confusion(matrix, names):\\n    a = matrix.astype(np.float64)\\n    diagonal = np.diag(a)\\n    union = a.sum(0) + a.sum(1) - diagonal\\n    total = a.sum(0) + a.sum(1)\\n    iou = [float(diagonal[i] / union[i]) if union[i] else None for i in range(5)]\\n    f1 = [float(2 * diagonal[i] / total[i]) if total[i] else None for i in range(5)]\\n    fg = [x for x in iou[1:] if x is not None]\\n    return dict(foreground_miou=float(np.mean(fg)) if fg else None,\\n                iou=dict(zip(names, iou)), f1=dict(zip(names, f1)),\\n                pixel_accuracy=float(diagonal.sum() / a.sum()) if a.sum() else None,\\n                confusion_matrix=matrix.tolist(), confusion_axes=\\"rows=annotation, columns=prediction\\")\\n", "baseline_review.py": "#!/usr/bin/env python3\\n\\"\\"\\"Audit the existing dataset or score an existing run\'s saved predictions.\\n\\nNo training or model download. Dependencies: numpy, rasterio, matplotlib.\\nThe saved prediction TIFFs, not screenshot pixels, are used for exact scoring.\\n\\"\\"\\"\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nimport math\\nfrom pathlib import Path\\nimport shutil\\nimport sys\\nimport tempfile\\n\\nimport numpy as np\\nimport rasterio\\n\\nfrom baseline_dataset_core import extract_dataset, inspect_dataset, metrics_from_confusion, sha256\\nfrom compare_per_class_iou import CLASSES, REF_FIELDS, make_reference, load_metrics, write_csv\\n\\n\\ndef write_json(path, data):\\n    Path(path).write_text(json.dumps(data, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef get_bundle(dataset, cache):\\n    dataset = Path(dataset)\\n    root = extract_dataset(dataset, cache) if dataset.is_file() else dataset\\n    bundle = inspect_dataset(root, input_bands=(1, 2, 3))\\n    if bundle[\'names\'] != list(CLASSES.values()):\\n        raise ValueError(\'Class names/order differ from the original five-output baseline.\')\\n    return bundle\\n\\n\\ndef new_output(path):\\n    path = Path(path)\\n    if path.exists() and any(path.iterdir()):\\n        raise ValueError(f\'Output is not empty: {path}. Choose a new folder to preserve earlier results.\')\\n    path.mkdir(parents=True, exist_ok=True)\\n    return path\\n\\n\\ndef pixel_hash(array):\\n    digest = hashlib.sha256()\\n    digest.update(str(array.shape).encode())\\n    digest.update(str(array.dtype).encode())\\n    digest.update(np.ascontiguousarray(array).tobytes())\\n    return digest.hexdigest()\\n\\n\\ndef audit_dataset(bundle, output, dataset=None):\\n    output = new_output(output)\\n    audit = dict(bundle[\'audit\'])\\n    audit[\'measurement_kind\'] = \'dataset_audit; not trained-model performance\'\\n    audit[\'source_archive_sha256\'] = sha256(dataset) if dataset and Path(dataset).is_file() else None\\n    hashes = {}\\n    for sample in bundle[\'samples\']:\\n        hashes.setdefault(pixel_hash(sample[\'image\']), []).append(sample[\'row\'])\\n    audit[\'duplicate_rgb_pixels_across_splits\'] = [\\n        [r[\'chip\'] for r in rows] for rows in hashes.values()\\n        if len({r[\'split\'] for r in rows}) > 1\\n    ]\\n    touching, minimum_gap = [], math.inf\\n    for a in bundle[\'splits\'][\'train\']:\\n        for b in bundle[\'splits\'][\'val\']:\\n            aa, bb = a[\'bounds\'], b[\'bounds\']\\n            dx = max(0., aa[0] - bb[2], bb[0] - aa[2])\\n            dy = max(0., aa[1] - bb[3], bb[1] - aa[3])\\n            gap = math.hypot(dx, dy)\\n            minimum_gap = min(minimum_gap, gap)\\n            if gap < 1e-4:\\n                touching.append([a[\'row\'][\'chip\'], b[\'row\'][\'chip\']])\\n    audit[\'touching_train_val_footprints\'] = touching\\n    audit[\'minimum_train_val_footprint_gap_m\'] = minimum_gap\\n    audit[\'geographic_caveat\'] = (\'No overlap/group collision does not establish independent geographic \'\\n                                  \'generalization; neighboring chips may share local appearance.\')\\n    spacings = [[abs(s[\'profile\'][\'transform\'].a), abs(s[\'profile\'][\'transform\'].e)]\\n                for s in bundle[\'samples\']]\\n    audit[\'exported_pixel_spacing_m_range\'] = {\\n        \'x\': [min(p[0] for p in spacings), max(p[0] for p in spacings)],\\n        \'y\': [min(p[1] for p in spacings), max(p[1] for p in spacings)]}\\n    audit[\'native_acquisition_gsd\'] = \'unknown; exported pixel spacing is not verified sensor GSD\'\\n    audit[\'input_bands_note\'] = \'Bands 1\\u20133 treated as RGB, matching the original baseline; band 4 unused.\'\\n    audit[\'validity_rule\'] = \'Ignore mask 255 and pixels invalid in any selected image band, as in the baseline.\'\\n    audit[\'checks_passed\'] = [\\n        \'unique chip IDs and existing image/mask pairs\', \'allowed label IDs 0\\u20134 and 255\',\\n        \'image/mask dimensions, CRS and affine alignment\', \'uint8 imagery and single-band integer masks\',\\n        \'nonempty train/validation sets and all foreground classes present in training\',\\n        \'no shared train/validation group IDs\', \'no positive-area train/validation footprint overlap\']\\n    if not audit[\'duplicate_rgb_pixels_across_splits\']:\\n        audit[\'checks_passed\'].append(\'no identical RGB pixel arrays across train/validation\')\\n    write_json(output/\'data_audit.json\', audit)\\n    write_csv(output/\'class_distribution.csv\', tuple(audit[\'class_report\'][0]), audit[\'class_report\'])\\n    write_csv(output/\'validation_reference.csv\', REF_FIELDS, make_reference(bundle[\'root\']))\\n    chip_rows = []\\n    for sample in bundle[\'samples\']:\\n        row = sample[\'row\']\\n        values = np.bincount(sample[\'mask\'][sample[\'mask\'] != 255], minlength=5)\\n        chip_rows.append({\'chip\': row[\'chip\'], \'split\': row[\'split\'],\\n                          \'label_status\': row.get(\'label_status\', \'unknown\'),\\n                          \'group\': row.get(\'group\', \'\'), \'valid_pixels\': int(values.sum()),\\n                          \'ignored_pixels\': int((sample[\'mask\'] == 255).sum()),\\n                          **{name+\'_pixels\': int(values[i]) for i, name in CLASSES.items()}})\\n    write_csv(output/\'per_chip_support.csv\', tuple(chip_rows[0]), chip_rows)\\n    return audit\\n\\n\\ndef accumulate(truth, prediction):\\n    if truth.shape != prediction.shape:\\n        raise ValueError(\'Prediction/mask shape mismatch.\')\\n    valid = truth != 255\\n    if not set(np.unique(prediction[valid]).tolist()).issubset(set(range(5))):\\n        raise ValueError(\'Prediction must contain a class ID 0\\u20134 at every evaluable pixel; 255 cannot hide errors.\')\\n    return np.bincount((truth[valid] * 5 + prediction[valid].astype(np.int64)).ravel(),\\n                       minlength=25).reshape(5, 5)\\n\\n\\ndef score_predictions(bundle, predictions):\\n    predictions = Path(predictions)\\n    confusion = np.zeros((5, 5), dtype=np.int64)\\n    chip_rows, prediction_refs = [], []\\n    for sample in bundle[\'splits\'][\'val\']:\\n        chip = sample[\'row\'][\'chip\']\\n        path = predictions/(chip+\'.tif\')\\n        if not path.is_file():\\n            raise ValueError(f\'Missing saved prediction: {path}\')\\n        with rasterio.open(path) as src:\\n            if src.count != 1 or not np.issubdtype(np.dtype(src.dtypes[0]), np.integer):\\n                raise ValueError(f\'{chip}: prediction must be a single-band integer TIFF.\')\\n            if (src.height, src.width) != sample[\'mask\'].shape or src.crs != sample[\'profile\'][\'crs\']:\\n                raise ValueError(f\'{chip}: prediction shape or CRS does not match the image.\')\\n            if not src.transform.almost_equals(sample[\'profile\'][\'transform\']):\\n                raise ValueError(f\'{chip}: prediction grid does not match the image.\')\\n            prediction = src.read(1)\\n        matrix = accumulate(sample[\'mask\'], prediction)\\n        confusion += matrix\\n        scores = metrics_from_confusion(matrix, bundle[\'names\'])\\n        chip_rows.append({\'chip\': chip, \'foreground_mIoU\': scores[\'foreground_miou\'],\\n                          **scores[\'iou\']})\\n        prediction_refs.append({\'chip\': chip, \'prediction_sha256\': sha256(path)})\\n    return confusion, chip_rows, prediction_refs\\n\\n\\ndef write_scores(bundle, confusion, chip_rows, prediction_refs, output, metadata):\\n    output = new_output(output)\\n    scores = metrics_from_confusion(confusion, bundle[\'names\'])\\n    write_json(output/\'metrics.json\', {**metadata, **scores})\\n    rows = []\\n    for i, name in CLASSES.items():\\n        tp, actual, predicted = int(confusion[i, i]), int(confusion[i].sum()), int(confusion[:, i].sum())\\n        rows.append({\'class\': name, \'IoU\': scores[\'iou\'][name], \'F1\': scores[\'f1\'][name],\\n                     \'precision\': tp/predicted if predicted else None,\\n                     \'recall\': tp/actual if actual else None,\\n                     \'label_pixels\': actual, \'predicted_pixels\': predicted})\\n    write_csv(output/\'per_class_metrics.csv\', tuple(rows[0]), rows)\\n    write_csv(output/\'per_chip_metrics.csv\', tuple(chip_rows[0]), chip_rows)\\n    write_csv(output/\'prediction_reference.csv\', tuple(prediction_refs[0]), prediction_refs)\\n    write_csv(output/\'validation_reference.csv\', REF_FIELDS, make_reference(bundle[\'root\']))\\n    matrix_rows = [{\'actual_class\': name, **{pred: int(confusion[i, j])\\n                    for j, pred in CLASSES.items()}} for i, name in CLASSES.items()]\\n    write_csv(output/\'confusion_matrix.csv\', tuple(matrix_rows[0]), matrix_rows)\\n    return scores\\n\\n\\ndef evaluate_run(bundle, run, output):\\n    run, output = Path(run), Path(output)\\n    original_audit = run/\'data_audit.json\'\\n    if not original_audit.is_file():\\n        raise ValueError(\'Run is missing data_audit.json; its dataset provenance cannot be checked.\')\\n    recorded_audit = json.loads(original_audit.read_text())\\n    if recorded_audit.get(\'dataset_fingerprint\') != bundle[\'audit\'][\'dataset_fingerprint\']:\\n        raise ValueError(\'Dataset differs from the saved run. Use that run\\\\\'s original dataset; \'\\n                         \'changed-label evaluation requires regenerating predictions/provenance deliberately.\')\\n    if recorded_audit.get(\'input_bands\') != [1, 2, 3]:\\n        raise ValueError(\'Saved run used different image bands; its valid-pixel rule needs a matching evaluator.\')\\n    old_metrics = json.loads((run/\'metrics.json\').read_text()) if (run/\'metrics.json\').is_file() else {}\\n    model_config = run/\'best_model/config.json\'\\n    if model_config.is_file():\\n        config = json.loads(model_config.read_text())\\n        if config.get(\'id2label\') != {str(k): v for k, v in CLASSES.items()}:\\n            raise ValueError(\'Saved model class mapping does not match the dataset.\')\\n    confusion, chip_rows, refs = score_predictions(bundle, run/\'predictions\')\\n    checkpoint_files = {}\\n    if (run/\'best_model\').is_dir():\\n        checkpoint_files = {str(p.relative_to(run)): sha256(p)\\n                            for p in sorted((run/\'best_model\').rglob(\'*\')) if p.is_file()}\\n    metadata = {\'measurement_kind\': \'recomputed from existing saved prediction TIFFs; no training\',\\n                \'source_run_name\': run.name, \'dataset_fingerprint\': bundle[\'audit\'][\'dataset_fingerprint\'],\\n                \'completed_epochs\': old_metrics.get(\'completed_epochs\'),\\n                \'best_epoch\': old_metrics.get(\'best_epoch\'),\\n                \'label_status\': bundle[\'audit\'][\'label_status\'],\\n                \'checkpoint_file_hashes\': checkpoint_files,\\n                \'metric_definition\': \'global pixel confusion matrix; mean of foreground class IoUs\',\\n                \'evaluation_caveat\': \'Agreement with supplied draft masks, not verified real-world accuracy.\'}\\n    scores = write_scores(bundle, confusion, chip_rows, refs, output, metadata)\\n    original_dir = output/\'original_run_files\'\\n    original_dir.mkdir()\\n    for name in [\'metrics.json\', \'per_class_metrics.csv\', \'per_chip_metrics.csv\', \'data_audit.json\',\\n                 \'training_config.json\', \'history.csv\', \'history.json\', \'TRAINING_NOTE.md\',\\n                 \'learning_curves.png\', \'qualitative_results.png\']:\\n        if (run/name).is_file():\\n            shutil.copy2(run/name, original_dir/name)\\n    if (run/\'per_class_metrics.csv\').is_file():\\n        old = load_metrics(run/\'per_class_metrics.csv\')\\n        rows = [{\'class\': name, \'original_logged_IoU\': old.get(name),\\n                 \'saved_prediction_IoU\': scores[\'iou\'][name],\\n                 \'difference\': scores[\'iou\'][name]-old[name] if name in old else None}\\n                for name in bundle[\'names\']]\\n        write_csv(output/\'original_vs_saved_prediction_scores.csv\', tuple(rows[0]), rows)\\n    (output/\'EVALUATION_NOTE.md\').write_text(\\n        \'# Existing baseline evaluation\\\\n\\\\n\'\\n        f\\"Scored {len(bundle[\'splits\'][\'val\'])} existing validation prediction TIFFs. No retraining.\\\\n\\\\n\\"\\n        f\\"Foreground mIoU against the supplied labels: **{scores[\'foreground_miou\']:.6f}**.\\\\n\\\\n\\"\\n        \'The original run files are preserved separately. Small differences from the logged \'\\n        \'validation scores may arise because training-time validation and exported prediction \'\\n        \'TIFFs used different inference batch sizes. Larger differences require investigation. \'\\n        \'Use the same saved-prediction scoring procedure for subsequent comparisons.\\\\n\\\\n\'\\n        \'If validation labels are corrected, score predictions from both checkpoints on the same \'\\n        \'corrected labels and record a new validation reference. The guarded original-run command \'\\n        \'intentionally refuses an unexplained dataset change.\\\\n\')\\n    return scores\\n\\n\\ndef run_controls(bundle, output):\\n    output = new_output(output)\\n    results = {}\\n    with tempfile.TemporaryDirectory(prefix=\'cloudtcp_controls_\') as temporary:\\n        temp = Path(temporary)\\n        for control in (\'all_background\', \'perfect_copy_of_labels\'):\\n            folder = temp/control\\n            folder.mkdir()\\n            for sample in bundle[\'splits\'][\'val\']:\\n                prediction = np.zeros(sample[\'mask\'].shape, dtype=np.uint8)\\n                if control == \'perfect_copy_of_labels\':\\n                    prediction = sample[\'mask\'].astype(np.uint8)\\n                prediction[sample[\'mask\'] == 255] = 255\\n                profile = dict(sample[\'profile\'])\\n                profile.update(driver=\'GTiff\', count=1, dtype=\'uint8\', nodata=255, compress=\'deflate\')\\n                profile.pop(\'photometric\', None)\\n                with rasterio.open(folder/(sample[\'row\'][\'chip\']+\'.tif\'), \'w\', **profile) as dst:\\n                    dst.write(prediction, 1)\\n            matrix, chips, refs = score_predictions(bundle, folder)\\n            result = write_scores(bundle, matrix, chips, refs, output/control,\\n                                  {\'measurement_kind\': \'sanity-check control; NOT a trained model\',\\n                                   \'control\': control})\\n            expected = 0. if control == \'all_background\' else 1.\\n            if abs(result[\'foreground_miou\'] - expected) > 1e-12:\\n                raise RuntimeError(f\'Scoring check failed: {control}\')\\n            results[control] = {\'foreground_mIoU\': result[\'foreground_miou\'],\\n                                \'pixel_accuracy\': result[\'pixel_accuracy\'], \'expected_foreground_mIoU\': expected}\\n    write_json(output/\'CONTROL_RESULTS.json\', {\\n        \'purpose\': \'Verify scoring on all 19 actual validation grids. These are not trained-model results.\',\\n        \'controls\': results})\\n    return results\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    commands = parser.add_subparsers(dest=\'command\', required=True)\\n    for name in (\'audit\', \'controls\', \'evaluate\'):\\n        command = commands.add_parser(name)\\n        command.add_argument(\'--dataset\', required=True, help=\'Dataset ZIP or dataset root folder\')\\n        command.add_argument(\'--output\', required=True, help=\'New output folder\')\\n        command.add_argument(\'--cache\', default=\'.cloudtcp_dataset_cache\')\\n        if name == \'evaluate\':\\n            command.add_argument(\'--run\', required=True, help=\'Existing run folder with prediction TIFFs\')\\n    args = parser.parse_args()\\n    try:\\n        bundle = get_bundle(args.dataset, args.cache)\\n        if args.command == \'audit\':\\n            result = audit_dataset(bundle, args.output, args.dataset)\\n            print(json.dumps({\'status\': \'passed\', \'n_chips\': result[\'n_chips\'],\\n                              \'foreground_train_pixels\': result[\'class_counts\'][\'train\'][1:],\\n                              \'minimum_train_val_footprint_gap_m\': result[\'minimum_train_val_footprint_gap_m\']}, indent=2))\\n        elif args.command == \'controls\':\\n            print(json.dumps(run_controls(bundle, args.output), indent=2))\\n        else:\\n            result = evaluate_run(bundle, args.run, args.output)\\n            print(json.dumps({\'foreground_mIoU\': result[\'foreground_miou\'], \'IoU\': result[\'iou\']}, indent=2))\\n    except (OSError, ValueError, KeyError) as exc:\\n        parser.exit(2, f\'Error: {exc}\\\\n\')\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "compare_per_class_iou.py": "#!/usr/bin/env python3\\n\\"\\"\\"Prepare validation references and compare CloudTCP per-class IoU CSVs.\\n\\nPython 3.9+; standard library only. This utility does not train or evaluate a\\nmodel. Save a reference at evaluation time; matching references cannot prove\\nthat independently supplied metric files were actually produced on that data.\\n\\"\\"\\"\\n\\nimport argparse\\nimport csv\\nimport hashlib\\nimport io\\nimport json\\nimport math\\nfrom pathlib import Path, PurePosixPath\\nimport re\\nimport sys\\nimport zipfile\\n\\n\\nCLASSES = {0: \\"background\\", 1: \\"curb\\", 2: \\"lane_marking\\",\\n           3: \\"crosswalk\\", 4: \\"median_boundary\\"}\\nFOREGROUND = tuple(CLASSES[i] for i in range(1, 5))\\nREF_FIELDS = (\\"chip\\", \\"split\\", \\"image\\", \\"mask\\", \\"image_sha256\\",\\n              \\"mask_sha256\\", \\"label_status\\")\\n\\n\\ndef csv_rows(text, required, source):\\n    reader = csv.DictReader(io.StringIO(text.lstrip(\\"\\\\ufeff\\")))\\n    fields = reader.fieldnames or []\\n    if len(fields) != len(set(fields)):\\n        raise ValueError(f\\"{source}: duplicate CSV column names\\")\\n    missing = set(required) - set(fields)\\n    if missing:\\n        raise ValueError(f\\"{source}: missing columns {sorted(missing)}\\")\\n    rows = list(reader)\\n    if not rows:\\n        raise ValueError(f\\"{source}: no data rows\\")\\n    if any(None in row or any(v is None for v in row.values()) for row in rows):\\n        raise ValueError(f\\"{source}: malformed CSV row\\")\\n    return rows\\n\\n\\ndef load_metrics(path):\\n    rows = csv_rows(Path(path).read_text(encoding=\\"utf-8-sig\\"),\\n                    (\\"class\\", \\"IoU\\"), path)\\n    result = {}\\n    for row in rows:\\n        name = row[\\"class\\"].strip()\\n        if name not in CLASSES.values():\\n            raise ValueError(f\\"{path}: unknown class {name!r}; use the original class names\\")\\n        if name in result:\\n            raise ValueError(f\\"{path}: duplicate class {name!r}\\")\\n        try:\\n            value = float(row[\\"IoU\\"])\\n        except ValueError as exc:\\n            raise ValueError(f\\"{path}: {name} has no numeric IoU; check class support\\") from exc\\n        if not math.isfinite(value) or not 0 <= value <= 1:\\n            raise ValueError(f\\"{path}: {name} IoU must be finite and in [0, 1], not a percentage\\")\\n        result[name] = value\\n    missing = set(FOREGROUND) - set(result)\\n    if missing:\\n        raise ValueError(f\\"{path}: missing foreground classes {sorted(missing)}\\")\\n    return result\\n\\n\\ndef safe_relative(name):\\n    path = PurePosixPath(name)\\n    if (not name or path.is_absolute() or \\"..\\" in path.parts\\n            or \\"\\\\\\\\\\" in name or \\":\\" in name or str(path) == \\".\\"):\\n        raise ValueError(f\\"Invalid dataset-relative path: {name!r}\\")\\n    return path\\n\\n\\ndef make_reference(dataset):\\n    dataset = Path(dataset)\\n    archive = None\\n    if dataset.is_file() and zipfile.is_zipfile(dataset):\\n        archive = zipfile.ZipFile(dataset)\\n        names = [n for n in archive.namelist()\\n                 if not n.endswith(\\"/\\") and \\"__MACOSX\\" not in PurePosixPath(n).parts]\\n        if len(names) != len(set(names)):\\n            archive.close()\\n            raise ValueError(\\"ZIP contains duplicate member names\\")\\n        roots = [str(PurePosixPath(n).parent) for n in names\\n                 if PurePosixPath(n).name == \\"manifest.csv\\"\\n                 and str(PurePosixPath(n).parent / \\"pipeline_settings.json\\") in names]\\n        if len(roots) != 1:\\n            archive.close()\\n            raise ValueError(\\"ZIP must have exactly one manifest/settings dataset root\\")\\n        root = PurePosixPath(roots[0])\\n\\n        def read_file(name):\\n            member = str(root / safe_relative(name))\\n            if member not in names:\\n                raise ValueError(f\\"Missing dataset file: {member}\\")\\n            return archive.read(member)\\n    elif dataset.is_dir():\\n        root = dataset.resolve()\\n\\n        def read_file(name):\\n            candidate = (root / safe_relative(name)).resolve()\\n            if root not in candidate.parents or not candidate.is_file():\\n                raise ValueError(f\\"Missing or outside-dataset file: {name}\\")\\n            return candidate.read_bytes()\\n    else:\\n        raise ValueError(\\"Dataset must be a ZIP or the folder containing manifest.csv\\")\\n\\n    try:\\n        settings = json.loads(read_file(\\"pipeline_settings.json\\"))\\n        expected = {str(k): v for k, v in CLASSES.items()}\\n        if settings.get(\\"classes\\") != expected or settings.get(\\"ignore_label\\") != 255:\\n            raise ValueError(\\"Expected the original class IDs 0\\u20134 and ignore_label=255\\")\\n        rows = csv_rows(read_file(\\"manifest.csv\\").decode(\\"utf-8-sig\\"),\\n                        (\\"chip\\", \\"split\\", \\"image\\", \\"mask\\"), \\"manifest.csv\\")\\n        seen, references = set(), []\\n        for row in rows:\\n            chip = row[\\"chip\\"].strip()\\n            if not chip or chip in seen:\\n                raise ValueError(f\\"Missing or duplicate chip ID: {chip!r}\\")\\n            seen.add(chip)\\n            if row[\\"split\\"] not in (\\"train\\", \\"val\\"):\\n                raise ValueError(f\\"{chip}: split must be train or val\\")\\n            if row[\\"split\\"] == \\"val\\":\\n                references.append({\\n                    \\"chip\\": chip, \\"split\\": \\"val\\", \\"image\\": row[\\"image\\"],\\n                    \\"mask\\": row[\\"mask\\"],\\n                    \\"image_sha256\\": hashlib.sha256(read_file(row[\\"image\\"])).hexdigest(),\\n                    \\"mask_sha256\\": hashlib.sha256(read_file(row[\\"mask\\"])).hexdigest(),\\n                    \\"label_status\\": row.get(\\"label_status\\", \\"unknown\\"),\\n                })\\n        if not references:\\n            raise ValueError(\\"No validation chips in manifest.csv\\")\\n        return sorted(references, key=lambda r: r[\\"chip\\"])\\n    finally:\\n        if archive is not None:\\n            archive.close()\\n\\n\\ndef load_reference(path):\\n    rows = csv_rows(Path(path).read_text(encoding=\\"utf-8-sig\\"),\\n                    (\\"chip\\", \\"split\\", \\"image_sha256\\", \\"mask_sha256\\"), path)\\n    result = {}\\n    for row in rows:\\n        chip = row[\\"chip\\"].strip()\\n        if not chip or chip in result:\\n            raise ValueError(f\\"{path}: missing or duplicate chip ID {chip!r}\\")\\n        if row[\\"split\\"] != \\"val\\":\\n            raise ValueError(f\\"{path}: {chip} is not a validation chip\\")\\n        hashes = tuple(row[field].strip().lower()\\n                       for field in (\\"image_sha256\\", \\"mask_sha256\\"))\\n        if any(not re.fullmatch(r\\"[0-9a-f]{64}\\", value) for value in hashes):\\n            raise ValueError(f\\"{path}: {chip} has an invalid SHA-256 hash\\")\\n        result[chip] = hashes\\n    return result\\n\\n\\ndef check_references(before, after):\\n    left, right = load_reference(before), load_reference(after)\\n    if set(left) != set(right):\\n        raise ValueError(\\"Validation chip IDs differ. Evaluate both checkpoints on one common set.\\")\\n    changed = sorted(chip for chip in left if left[chip] != right[chip])\\n    if changed:\\n        raise ValueError(\\"Validation image or mask bytes changed for \\" + \\", \\".join(changed)\\n                         + \\". Re-evaluate BOTH checkpoints on the same images and masks.\\")\\n    return len(left)\\n\\n\\ndef write_csv(path, fields, rows):\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    # Refuse to overwrite an earlier report or an input file accidentally.\\n    with path.open(\\"x\\", encoding=\\"utf-8\\", newline=\\"\\") as handle:\\n        writer = csv.DictWriter(handle, fieldnames=fields)\\n        writer.writeheader()\\n        writer.writerows(rows)\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    commands = parser.add_subparsers(dest=\\"command\\", required=True)\\n    reference = commands.add_parser(\\"reference\\", help=\\"Hash the current validation images/masks\\")\\n    reference.add_argument(\\"--dataset\\", required=True)\\n    reference.add_argument(\\"--output\\", required=True)\\n    rank = commands.add_parser(\\"rank\\", help=\\"Rank the four foreground classes by increasing IoU\\")\\n    rank.add_argument(\\"metrics\\", help=\\"Existing per_class_metrics.csv\\")\\n    rank.add_argument(\\"--output\\", required=True)\\n    compare = commands.add_parser(\\"compare\\", help=\\"Compare IoU on a matching validation reference\\")\\n    compare.add_argument(\\"before\\")\\n    compare.add_argument(\\"after\\")\\n    compare.add_argument(\\"--before-eval-reference\\", required=True)\\n    compare.add_argument(\\"--after-eval-reference\\", required=True)\\n    compare.add_argument(\\"--output\\", required=True)\\n    args = parser.parse_args()\\n    try:\\n        if args.command == \\"reference\\":\\n            rows = make_reference(args.dataset)\\n            write_csv(args.output, REF_FIELDS, rows)\\n            print(f\\"Recorded {len(rows)} validation chips. Save this file with the matching evaluation.\\")\\n        elif args.command == \\"rank\\":\\n            metrics = load_metrics(args.metrics)\\n            ordered = sorted(FOREGROUND, key=lambda name: (metrics[name], name))\\n            rows = [{\\"rank\\": i, \\"class\\": name, \\"IoU\\": f\\"{metrics[name]:.8f}\\"}\\n                    for i, name in enumerate(ordered, start=1)]\\n            write_csv(args.output, (\\"rank\\", \\"class\\", \\"IoU\\"), rows)\\n            print(\\"Lowest IoU first: \\" + \\", \\".join(f\\"{n}={metrics[n]:.4f}\\" for n in ordered))\\n        else:\\n            before, after = load_metrics(args.before), load_metrics(args.after)\\n            n_chips = check_references(args.before_eval_reference, args.after_eval_reference)\\n            rows = []\\n            for name in FOREGROUND:\\n                delta = after[name] - before[name]\\n                rows.append({\\"class\\": name, \\"before_IoU\\": f\\"{before[name]:.8f}\\",\\n                             \\"after_IoU\\": f\\"{after[name]:.8f}\\", \\"delta_IoU\\": f\\"{delta:.8f}\\",\\n                             \\"delta_percentage_points\\": f\\"{100 * delta:.6f}\\"})\\n            mean_before = sum(before[n] for n in FOREGROUND) / len(FOREGROUND)\\n            mean_after = sum(after[n] for n in FOREGROUND) / len(FOREGROUND)\\n            rows.append({\\"class\\": \\"foreground_mIoU\\", \\"before_IoU\\": f\\"{mean_before:.8f}\\",\\n                         \\"after_IoU\\": f\\"{mean_after:.8f}\\",\\n                         \\"delta_IoU\\": f\\"{mean_after - mean_before:.8f}\\",\\n                         \\"delta_percentage_points\\": f\\"{100 * (mean_after - mean_before):.6f}\\"})\\n            write_csv(args.output, tuple(rows[0]), rows)\\n            print(f\\"Compared 4 foreground classes; {n_chips} validation image/mask pairs match.\\")\\n            print(f\\"Foreground mIoU: {mean_before:.4f} -> {mean_after:.4f} \\"\\n                  f\\"({100 * (mean_after - mean_before):+.2f} percentage points)\\")\\n            print(\\"Hashes check supplied references only; verify metric/checkpoint provenance separately.\\")\\n    except (ValueError, OSError, KeyError, csv.Error, zipfile.BadZipFile) as exc:\\n        parser.exit(2, f\\"Error: {exc}\\\\n\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for name, content in EMBEDDED_FILES.items():
    (TOOL_DIR / name).write_text(content)
print("Evaluation code ready.")


## 3. Find the dataset and saved run

Keep `INPUT_MODE = "drive"` to use Google Drive. If several runs are found, set `RUN_NAME` to the 40-epoch run you want to evaluate and rerun the cell.

For local uploads, set `INPUT_MODE = "upload"` and select both `dataset_split.zip` and the original results ZIP. This mode does not use Drive.


In [ ]:
from datetime import datetime, timezone
import zipfile
from google.colab import files

INPUT_MODE = "drive"  # "drive" or "upload"
RUN_NAME = ""  # Leave blank to select a unique 40-epoch run automatically.
PROJECT_DIR = Path("/content/drive/MyDrive/CloudTCP")
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
WORK = Path("/content") / ("cloudtcp_baseline_" + STAMP)
WORK.mkdir()

if INPUT_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET = PROJECT_DIR / "dataset_split.zip"
    if not DATASET.is_file():
        raise FileNotFoundError(f"Dataset not found at {DATASET}. Place it there or use upload mode.")
    run_parent = PROJECT_DIR / "runs"
    candidates = []
    if run_parent.is_dir():
        for folder in sorted(run_parent.iterdir()):
            if folder.is_dir() and (folder / "metrics.json").is_file():
                metadata = json.loads((folder / "metrics.json").read_text())
                candidates.append((folder, metadata))
    print("Saved runs:")
    for folder, metadata in candidates:
        print(folder.name, "completed epochs:", metadata.get("completed_epochs"),
              "best epoch:", metadata.get("best_epoch"))
    if RUN_NAME:
        if Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
            raise ValueError("RUN_NAME must be one folder name, not a path.")
        RUN = run_parent / RUN_NAME
    else:
        matching = [folder for folder, metadata in candidates if metadata.get("completed_epochs") == 40]
        if len(matching) != 1:
            raise ValueError("Set RUN_NAME to your intended baseline run from the list above, then rerun this cell.")
        RUN = matching[0]
elif INPUT_MODE == "upload":
    print("Select your original dataset ZIP AND training results ZIP.")
    uploaded = files.upload()
    dataset_zips, result_zips = [], []
    for name, data in uploaded.items():
        destination = WORK / Path(name).name
        destination.write_bytes(data)
        if not zipfile.is_zipfile(destination):
            continue
        with zipfile.ZipFile(destination) as archive:
            names = [n for n in archive.namelist() if "__MACOSX" not in Path(n).parts]
            if any(Path(n).name == "manifest.csv" for n in names):
                dataset_zips.append(destination)
            elif any(Path(n).name == "metrics.json" for n in names):
                result_zips.append(destination)
    del uploaded
    if len(dataset_zips) != 1 or len(result_zips) != 1:
        raise ValueError("Expected exactly one original dataset ZIP and one training results ZIP.")
    DATASET = dataset_zips[0]
    result_root = WORK / "uploaded_run"
    result_root.mkdir()
    with zipfile.ZipFile(result_zips[0]) as archive:
        for member in archive.infolist():
            if "__MACOSX" in Path(member.filename).parts:
                continue
            target = (result_root / member.filename).resolve()
            if not target.is_relative_to(result_root.resolve()):
                raise ValueError("Unsafe path in results ZIP.")
            archive.extract(member, result_root)
    roots = [path.parent for path in result_root.rglob("metrics.json")
             if (path.parent / "data_audit.json").is_file()
             and (path.parent / "predictions").is_dir()]
    if len(roots) != 1:
        raise ValueError("Results ZIP must contain one run with data_audit.json and predictions/.")
    RUN = roots[0]
else:
    raise ValueError("INPUT_MODE must be 'drive' or 'upload'.")

for needed in ("data_audit.json", "metrics.json", "predictions"):
    if not (RUN / needed).exists():
        raise FileNotFoundError(f"Saved run is missing {needed}. Use the original exported results ZIP.")
print("Dataset:", DATASET)
print("Run:", RUN)
print("Recorded completed epochs:", json.loads((RUN / "metrics.json").read_text()).get("completed_epochs"))


## 4. Calculate scores

The code checks that the dataset matches the run, reads the 19 validation prediction TIFFs, and calculates per-class IoU across their pixels. It keeps the original logs separately and records any difference between their scores and the TIFF-based scores.


In [ ]:
import subprocess, sys
EVALUATION = WORK / "baseline_evaluation"
subprocess.run([
    sys.executable, str(TOOL_DIR / "baseline_review.py"), "evaluate",
    "--dataset", str(DATASET), "--run", str(RUN),
    "--output", str(EVALUATION), "--cache", str(WORK / "dataset_cache"),
], check=True)
subprocess.run([
    sys.executable, str(TOOL_DIR / "compare_per_class_iou.py"), "rank",
    str(EVALUATION / "per_class_metrics.csv"),
    "--output", str(EVALUATION / "weak_classes.csv"),
], check=True)

import pandas as pd
from IPython.display import display
print("Scores from the EXISTING trained model's saved predictions:")
display(pd.read_csv(EVALUATION / "per_class_metrics.csv"))
print("Foreground classes ranked from lowest IoU:")
display(pd.read_csv(EVALUATION / "weak_classes.csv"))


## 5. Check the comparison

Compare the baseline with itself. Every change should be zero. This tests the comparison code, not model improvement. When a new model is ready, use its evaluation as the second input.


In [ ]:
subprocess.run([
    sys.executable, str(TOOL_DIR / "compare_per_class_iou.py"), "compare",
    str(EVALUATION / "per_class_metrics.csv"), str(EVALUATION / "per_class_metrics.csv"),
    "--before-eval-reference", str(EVALUATION / "validation_reference.csv"),
    "--after-eval-reference", str(EVALUATION / "validation_reference.csv"),
    "--output", str(EVALUATION / "same_baseline_workflow_check.csv"),
], check=True)
check = pd.read_csv(EVALUATION / "same_baseline_workflow_check.csv")
assert (check["delta_IoU"] == 0).all()
display(check)


## 6. Save and download

The ZIP contains metrics, class rankings, validation-file hashes, logs, and figures. The model checkpoint stays where it is. Its hashes are recorded if the file was available.


In [ ]:
import shutil
archive_path = Path(shutil.make_archive(str(WORK / ("CloudTCP_W4_Exact_Baseline_" + STAMP)), "zip", EVALUATION))
if INPUT_MODE == "drive":
    destination = PROJECT_DIR / "baseline_reviews"
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive_path, destination / archive_path.name)
    print("Saved to:", destination / archive_path.name)
files.download(str(archive_path))


## Compare with a later run

Use the same validation images, masks, class IDs, and scoring rules. Evaluate each run with its matching dataset archive, then compare the evaluation folders with `compare_per_class_iou.py compare`.

If labels have been corrected, score both models against those same corrected labels.
